<img src="https://udemedellin.edu.co/wp-content/uploads/2022/10/logo_udemedellin2.png" width="30%">

<b>ESPECIALIZACIÓN EN CIENCIA DE DATOS E INGELIGENCIA ARTIFICIAL</b>

<strong>Fundamentos de Estadística para Ciencia de Datos</strong>

# Sesión 05 — Taller práctico: estadística descriptiva (caso de estudio)

## Objetivos de aprendizaje

Al finalizar este taller serás capaz de:

- Aplicar el flujo completo de estadística descriptiva (tipos de datos, calidad de datos, descriptiva cualitativa y cuantitativa) sobre un caso real, integrando lo visto en las sesiones 1 a 4.
- Tomar y justificar decisiones de limpieza de datos (faltantes) según el contexto de cada variable, incluyendo cuándo la ausencia de un dato puede ser informativa y no aleatoria.
- Construir variables derivadas (*feature engineering*) cuando aportan una señal que las variables originales no dan por separado.
- Detectar y tratar atípicos (no solo detectarlos): comparar recorte (*capping*) contra transformación logarítmica según el objetivo del análisis.
- Describir variables cualitativas y cuantitativas, y relacionarlas con una variable target de negocio, sin extrapolar más allá de lo que los datos permiten.

> **Nota:** este taller integra las sesiones 1 a 4. No se abordan pruebas de hipótesis ni modelado — eso es contenido de la sesión 6 y siguientes.

> **Nota:** este es el notebook para construir en vivo durante la clase. Las celdas de código quedan con `# TODO` a propósito, y las conclusiones de cada bloque están planteadas como preguntas guía en vez de respuestas. Existe una versión resuelta de referencia (`05_taller_practico_estadistica_descriptiva_resuelto.ipynb`).

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", palette="deep")
%matplotlib inline

pd.set_option("display.precision", 3)

## 1. Planteamiento del caso y carga de datos

**Contexto:** eres analista de riesgo en una entidad financiera. El equipo de crédito quiere entender qué caracteriza a las solicitudes de préstamo aprobadas frente a las rechazadas, antes de construir cualquier modelo de aprobación automática.

**Objetivo del taller:** dejar el dataset `loan` explorado, limpio y descrito (variables cualitativas y cuantitativas), identificando qué caracteriza a las solicitudes según su estado de aprobación, como insumo para decisiones posteriores. No se construye ningún modelo ni prueba de hipótesis en este taller.

**Fuente:** dataset público "Loan Prediction" (Analytics Vidhya / dphi-official), cargado directamente por URL.

In [2]:
# TODO: carga loan con pd.read_csv desde
# "https://raw.githubusercontent.com/dphi-official/Datasets/master/Loan_Data/loan_train.csv"
# (usa index_col=0)
# TODO: imprime shape y revisa las primeras filas con .head()

In [3]:
# TODO: usa .info() para revisar tipos de dato y conteo de no nulos por columna

### Diccionario de datos

El repositorio fuente no trae documentación propia (es un mirror sin README). Este es el diccionario del dataset original "Loan Prediction Practice Problem" (Analytics Vidhya), reconocible por ser un dataset público muy replicado — documentado igual en numerosos tutoriales y mirrors de Kaggle que usan el mismo problema.

| Variable | Significado |
|---|---|
| `Loan_ID` | ID único de la solicitud |
| `Gender` | Género del solicitante |
| `Married` | Estado civil del solicitante (casado/no) |
| `Dependents` | Número de dependientes económicos |
| `Education` | Nivel educativo (Graduate / Not Graduate) |
| `Self_Employed` | Si el solicitante es independiente |
| `ApplicantIncome` | Ingreso mensual del solicitante |
| `CoapplicantIncome` | Ingreso mensual del coaplicante (0 si no hay) |
| `LoanAmount` | Monto del préstamo solicitado, en miles |
| `Loan_Amount_Term` | Plazo del préstamo, en meses |
| `Credit_History` | Si el historial crediticio cumple los criterios del banco (1) o no (0) |
| `Property_Area` | Zona del inmueble (Urban / Semiurban / Rural) |
| `Loan_Status` | Si el préstamo fue aprobado (Y/N, aquí 1/0) — **variable target** |

### Clasificación de variables

Clasifica cada columna según la taxonomía de la sesión 1 (cualitativa nominal/ordinal, cuantitativa discreta/continua).

`Loan_ID`, `Gender`, `Married`, `Dependents`, `Education`, `Self_Employed`, `ApplicantIncome`, `CoapplicantIncome`, `LoanAmount`, `Loan_Amount_Term`, `Credit_History`, `Property_Area`, `Loan_Status`

**Pregunta guía:** `Dependents` y `Credit_History` no son tan directas como el resto. `Dependents` en el fondo es un conteo, pero el valor `"3+"` agrupa "3 o más" en una categoría abierta — ¿tiene sentido forzarla a numérica, o conviene tratarla como ordinal? `Credit_History` es numérica (0/1) pero, ¿tiene sentido tratarla como cuantitativa continua? ¿Y `Loan_ID`, aporta algo al análisis?

In [4]:
# TODO: elimina la columna Loan_ID (no aporta información analítica)

## 2. Calidad de datos: valores faltantes

Este dataset no tiene valores físicamente imposibles. El único problema de calidad es la ausencia de datos.

In [5]:
# TODO: cuenta valores faltantes por columna con .isna().sum()

**Pregunta guía:** ¿qué columna tiene más faltantes? Antes de imputarla, ¿tiene sentido revisar si esos faltantes se comportan distinto frente al target `Loan_Status`, o distinto sería asumir que la ausencia es aleatoria sin comprobarlo?

In [6]:
# TODO: calcula la tasa de aprobación general (loan["Loan_Status"].mean())
# TODO: calcula la tasa de aprobación por categoría de Credit_History (groupby)
# TODO: calcula la tasa de aprobación cuando Credit_History es NaN

**Pregunta guía:** ¿la tasa de aprobación del subgrupo con `Credit_History` faltante se parece más al grupo con historial en regla o al grupo sin él? ¿Qué te dice eso sobre si es defendible imputar con la moda?

`Dependents` tiene una categoría abierta, `"3+"`, que agrupa "3 o más". ¿Vale la pena convertirla a numérica para este taller, o alcanza con tratarla como categórica ordinal e imputar la moda igual que las demás?

In [7]:
# TODO: grafica la distribución de las 7 variables con faltantes antes de imputar:
#   Gender, Married, Self_Employed, Dependents, Loan_Amount_Term, Credit_History (countplot)
#   y LoanAmount (histograma) — te ayuda a decidir qué valor usar para imputar cada una

In [8]:
# TODO: construye clean_loan aplicando las decisiones que acabas de justificar:
#   1. imputa Gender, Married, Self_Employed, Dependents, Loan_Amount_Term, Credit_History con la moda
#   2. imputa LoanAmount con la mediana (¿por qué mediana y no media, dado lo que sabes de su distribución?)
# TODO: verifica que no queden NaN

## 3. Feature engineering

Con los datos limpios, construye dos variables derivadas con sentido de negocio directo en evaluación de riesgo crediticio.

**Pregunta guía:** ¿por qué sumar `ApplicantIncome` y `CoapplicantIncome` puede aportar más que `ApplicantIncome` solo? (pista: revisa cuántos coaplicantes tienen `CoapplicantIncome == 0`). ¿Y qué información captura `LoanAmount` dividido entre el ingreso total, que ninguna de las dos variables originales captura por separado? Revisa el diccionario de datos antes de dividir: ¿`LoanAmount` está en las mismas unidades que `ApplicantIncome`/`CoapplicantIncome`?

In [9]:
# TODO: crea TotalIncome = ApplicantIncome + CoapplicantIncome
# TODO: crea LoanAmount_to_Income = LoanAmount / TotalIncome, corrigiendo la diferencia de unidades
#   (LoanAmount está en miles, ver diccionario de datos)

## 4. Descriptiva de variables cualitativas

Describe `Gender`, `Married`, `Education` y `Property_Area`, relacionándolas con `Loan_Status`.

In [10]:
# TODO: tabla de frecuencias (FA, FR, %) de Gender

In [11]:
# TODO: grafica un countplot de Gender, Married, Education y Property_Area (4 subplots)

In [12]:
# TODO: para cada una de Gender, Married, Education, Property_Area:
#   tabla de contingencia contra Loan_Status, normalizada por fila (normalize="index")

In [13]:
# TODO: grafica un gráfico de barras 100% apiladas (Loan_Status) para cada una de
#   Gender, Married, Education, Property_Area (4 subplots) — usa las tablas de contingencia
#   que acabas de calcular y .plot(kind="bar", stacked=True)

**Pregunta guía:** ¿qué categoría de cada variable tiene mayor tasa de aprobación? ¿alguna diferencia te parece grande, o son todas relativamente similares a la tasa general?

## 5. Descriptiva de variables cuantitativas

Usa `ApplicantIncome` como variable "hilo conductor" de esta sección (igual que `horsepower` en las sesiones 2 y 3).

In [14]:
# TODO: ApplicantIncome.describe()

In [15]:
# TODO: calcula media, mediana, desviación estándar, coeficiente de variación, Q1, Q3 e IQR de ApplicantIncome

**Pregunta guía:** ¿qué te dice un coeficiente de variación así de alto sobre la dispersión relativa de `ApplicantIncome`?

In [16]:
# TODO: histograma + KDE de ApplicantIncome, y boxplot de ApplicantIncome (dos subplots)

### Evaluación de normalidad de `ApplicantIncome`

In [17]:
# TODO: calcula asimetría (stats.skew) y curtosis (stats.kurtosis) de ApplicantIncome
# TODO: aplica stats.shapiro e interpreta el p-valor
# TODO: grafica el QQ-plot con stats.probplot

**Pregunta guía:** ¿qué tan asimétrica es `ApplicantIncome` comparada con `horsepower` (~1.1) en la sesión 3?

## 6. Tratamiento de atípicos

Hasta ahora solo detectamos atípicos con la regla del IQR. Aquí vamos un paso más allá: **¿qué hacer con ellos?**

**Pregunta guía:** ¿por qué eliminar las filas con ingresos atípicos no sería defendible en este caso?

In [18]:
# TODO: para ApplicantIncome y LoanAmount, calcula el límite superior de atípicos con la regla del IQR
# TODO: cuenta cuántos valores superan ese límite en cada variable, y compáralo con el máximo de cada una

In [19]:
# TODO: crea ApplicantIncome_capped y LoanAmount_capped, recortando (clip) al límite superior del IQR
# TODO: crea una versión log-transformada de ApplicantIncome con np.log1p
# TODO: compara la asimetría (stats.skew) de: la variable original, la versión con capping, y la versión log

**Pregunta guía:** ¿cuál de las dos estrategias (capping o log) reduce más la asimetría? ¿En qué escenario elegirías capping en vez de log, aunque reduzca menos la asimetría? (pista: piensa en si necesitas seguir interpretando la variable en sus unidades originales).

## 7. Relación con la variable target (`Loan_Status`)

Cierra comparando las variables numéricas (incluyendo las tratadas y las derivadas) frente al estado de aprobación.

In [20]:
# TODO: boxplot de ApplicantIncome_capped, LoanAmount_capped y TotalIncome, cada uno agrupado por Loan_Status (3 subplots)

In [21]:
# TODO: tabla resumen (mean, median) de ApplicantIncome_capped, LoanAmount_capped, TotalIncome y LoanAmount_to_Income, agrupadas por Loan_Status

**Pregunta guía:** ¿esperabas que un ingreso más alto facilitara la aprobación? ¿Es eso lo que muestran los datos? ¿Qué te dice esto sobre confiar en la intuición de negocio sin contrastarla?

In [22]:
# TODO: tabla de contingencia Credit_History x Loan_Status, normalizada por fila

**Pregunta guía:** compara la magnitud de la diferencia en tasa de aprobación por `Credit_History` contra la de `ApplicantIncome` o cualquier variable cualitativa que hayas descrito. ¿Cuál domina?

## Cierre

Con base en lo que fuiste construyendo hoy, en grupo:

- ¿Qué decisiones de limpieza tomaste y por qué? ¿En qué caso revisaste si el faltante estaba relacionado con el target antes de imputar?
- ¿Las variables que construiste (`TotalIncome`, `LoanAmount_to_Income`) aportaron la señal que esperabas?
- ¿Cuándo usarías capping y cuándo transformación logarítmica para tratar atípicos?
- ¿Cuál fue el hallazgo que más te sorprendió, y qué tan seguro estás de que es un patrón real y no un artefacto de esta muestra en particular?
- Si tuvieras que confirmar formalmente si `Credit_History` está *realmente* asociada a `Loan_Status` (y no es azar), ¿qué necesitarías? (a definir con más profundidad en la sesión 6 en adelante)